# Foundations for ML: returns, risk, benchmarks, data and the statistics models use

The five pages of the textbook's *Foundations for ML* part, run in code. Open **one** notebook, run the setup cell, then any section; each matches a page, in order.

**No API key and nothing to download.** Everything runs on numpy, pandas and matplotlib. The figures are **illustrative**, made up for teaching (Deere, Caterpillar, a market index and three factor benchmarks), not market data.

| Section | Page | What you run |
|---|---|---|
| 1 | F1 | Simple, log and excess returns; compounding |
| 2 | F2 | Volatility, drawdown, Sharpe ratio |
| 3 | F3 | Beta and alpha against the market, and against three factors |
| 4 | F4 | Valuation ratios; what a model could know on a given day |
| 5 | F5 | A fitted line, R squared, and a train/test split by time |

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt

# --- the same illustrative data and helpers the textbook's in-page cells use (finance_basics.py) ---
import math

MONTHS = ["Oct 24", "Nov 24", "Dec 24", "Jan 25", "Feb 25", "Mar 25", "Apr 25", "May 25", "Jun 25", "Jul 25", "Aug 25", "Sep 25",
          "Oct 25", "Nov 25", "Dec 25", "Jan 26", "Feb 26", "Mar 26", "Apr 26", "May 26", "Jun 26", "Jul 26", "Aug 26", "Sep 26"]

# Monthly returns, as fractions (0.0086 = +0.86%).
MARKET = [0.0086, -0.0166, 0.0233, 0.0585, 0.0706, 0.032, 0.01, -0.0021, -0.0093, -0.032, -0.0531, -0.0618, 0.0478, 0.0058, 0.0462, -0.0464, 0.018, 0.0074, 0.0339, -0.0012, 0.0442, 0.0462, 0.0594, -0.0718]
SIZE = [-0.0237, 0.0292, 0.0526, 0.0111, -0.0058, 0.0251, 0.0053, 0.0052, 0.028, 0.0023, -0.0331, -0.0128, 0.0135, -0.0159, 0.0087, -0.01, 0.0121, -0.0093, 0.0193, -0.0063, -0.0207, 0.0116, 0.0084, 0.0039]
VALUE = [-0.036, -0.009, 0.0265, 0.0091, -0.0278, 0.0338, 0.0129, 0.0013, -0.0093, -0.0012, 0.0146, -0.0564, -0.0069, -0.0298, 0.0129, 0.0048, 0.0027, 0.0123, 0.0442, 0.0491, -0.0294, 0.0465, 0.0049, -0.0411]
MOMENTUM = [0.0436, -0.0202, -0.0155, 0.0316, -0.0019, 0.0089, 0.0414, 0.0146, -0.0756, -0.0074, -0.0363, -0.0341, -0.0352, 0.0098, 0.0072, -0.0032, 0.0356, 0.0078, -0.0043, 0.0346, 0.0208, 0.0111, 0.0375, 0.0278]
RISK_FREE = [0.0035] * 24   # about 4.3% a year, paid monthly

RETURNS = {
    "DE": [0.0002, -0.0156, 0.0303, 0.0429, 0.143, 0.0623, -0.0545, 0.0035, -0.0284, -0.0655, -0.0669, -0.0805, 0.0887, -0.0112, 0.0538, -0.0353, -0.0482, -0.0356, 0.0137, 0.0464, 0.0277, 0.0372, 0.0427, -0.1365],
    "CAT": [0.0086, -0.0313, 0.0492, 0.0959, 0.1035, 0.0601, -0.0025, 0.0047, -0.0242, -0.0444, -0.0278, -0.1255, 0.0728, 0.0096, 0.063, -0.0777, 0.0231, 0.0034, 0.0224, -0.0186, -0.004, 0.0588, 0.1594, -0.1386],
    "MKT": MARKET,
}
# Month-end price before the first return, so that the last price is today's price in tools.PRICES (DE 512.40, CAT 398.15).
START = {"DE": 527.71, "CAT": 331.01, "MKT": 100.0}


def prices(ticker):
    """The 25 month-end prices (Sep 24 to Sep 26) built from START and the monthly returns."""
    p = [START[ticker]]
    for r in RETURNS[ticker]:
        p.append(round(p[-1] * (1 + r), 2))
    return p


def simple_returns(p):
    """Simple return: the price change as a fraction of the old price."""
    return [p[i + 1] / p[i] - 1 for i in range(len(p) - 1)]


def log_returns(p):
    """Log return: ln(new price / old price). Adds up across months; simple returns do not."""
    return [math.log(p[i + 1] / p[i]) for i in range(len(p) - 1)]


def compound(returns):
    """Total return over the whole period: multiply (1 + r) across months, then subtract 1."""
    t = 1.0
    for r in returns:
        t *= 1 + r
    return t - 1


def excess(returns, rf=RISK_FREE):
    """Excess return: what you earned over what cash would have paid."""
    return [r - f for r, f in zip(returns, rf)]


def mean(x):
    return sum(x) / len(x)


def stdev(x):
    """Sample standard deviation (divides by n - 1)."""
    m = mean(x)
    return math.sqrt(sum((v - m) ** 2 for v in x) / (len(x) - 1))


def annual_return(returns):
    """Compounded return scaled to a year, from monthly returns."""
    return (1 + compound(returns)) ** (12 / len(returns)) - 1


def annual_vol(returns):
    """Volatility: the monthly standard deviation scaled to a year by the square root of 12."""
    return stdev(returns) * math.sqrt(12)


def drawdowns(p):
    """At each date, how far the price is below its highest earlier price (0 at a new high, else negative)."""
    peak, out = p[0], []
    for v in p:
        peak = max(peak, v)
        out.append(v / peak - 1)
    return out


def max_drawdown(p):
    """The worst peak-to-trough fall: (depth, index of the peak, index of the trough)."""
    peak_i, worst, w_peak, w_trough = 0, 0.0, 0, 0
    for i, v in enumerate(p):
        if v > p[peak_i]:
            peak_i = i
        d = v / p[peak_i] - 1
        if d < worst:
            worst, w_peak, w_trough = d, peak_i, i
    return worst, w_peak, w_trough


def sharpe(returns, rf=RISK_FREE):
    """Sharpe ratio: average excess return per unit of volatility, scaled to a year."""
    e = excess(returns, rf)
    return mean(e) / stdev(e) * math.sqrt(12)


def corr(x, y):
    mx, my = mean(x), mean(y)
    sxy = sum((a - mx) * (b - my) for a, b in zip(x, y))
    return sxy / math.sqrt(sum((a - mx) ** 2 for a in x) * sum((b - my) ** 2 for b in y))


def ols(x, y):
    """Fit y = intercept + slope * x by least squares: (intercept, slope, r_squared)."""
    mx, my = mean(x), mean(y)
    slope = sum((a - mx) * (b - my) for a, b in zip(x, y)) / sum((a - mx) ** 2 for a in x)
    intercept = my - slope * mx
    ss_res = sum((b - intercept - slope * a) ** 2 for a, b in zip(x, y))
    ss_tot = sum((b - my) ** 2 for b in y)
    return intercept, slope, 1 - ss_res / ss_tot


def alpha_beta(stock, market=MARKET, rf=RISK_FREE):
    """Regress the stock's excess return on the market's: alpha (a month), beta, r_squared."""
    return ols(excess(market, rf), excess(stock, rf))


def split_by_time(x, frac=0.75):
    """Train on the early part, test on the later part. Never shuffle a time series."""
    k = int(len(x) * frac)
    return x[:k], x[k:]


# Valuation: price over the last twelve months of earnings per share (tools.PRICES for the price).
FUNDAMENTALS = {
    "DE": {"price": 512.40, "eps_ttm": 25.60, "book_per_share": 71.20, "dividend": 6.40},
    "CAT": {"price": 398.15, "eps_ttm": 21.90, "book_per_share": 44.80, "dividend": 5.80},
}


def valuation(ticker):
    """Price-to-earnings, price-to-book and dividend yield from FUNDAMENTALS."""
    f = FUNDAMENTALS[ticker]
    return {"pe": f["price"] / f["eps_ttm"], "pb": f["price"] / f["book_per_share"], "yield": f["dividend"] / f["price"]}


# Macro releases: what a model could have known on a given day. Each row is one figure, the date it was first
# published, the first value, and (if revised) the date and value of the revision. Illustrative.
MACRO = [
    {"series": "CPI inflation, year on year", "period": "Aug 26", "freq": "monthly", "first": "2026-09-11", "value": 3.1, "revised": None, "revised_value": None},
    {"series": "Unemployment rate", "period": "Aug 26", "freq": "monthly", "first": "2026-09-04", "value": 4.3, "revised": None, "revised_value": None},
    {"series": "Nonfarm payrolls, change (thousands)", "period": "Aug 26", "freq": "monthly", "first": "2026-09-04", "value": 142, "revised": "2026-10-02", "revised_value": 96},
    {"series": "Real GDP growth, annualized", "period": "Q2 26", "freq": "quarterly", "first": "2026-07-30", "value": 2.6, "revised": "2026-08-27", "revised_value": 2.1},
    {"series": "Fed funds target, upper bound", "period": "Sep 26", "freq": "8 meetings a year", "first": "2026-09-16", "value": 4.50, "revised": None, "revised_value": None},
]


def known_on(series, date):
    """The value a model could have used on `date` (YYYY-MM-DD): the revised figure only after its revision date."""
    for m in MACRO:
        if m["series"] == series:
            if date < m["first"]:
                return None
            if m["revised"] and date >= m["revised"]:
                return m["revised_value"]
            return m["value"]
    raise KeyError(series)

## 1 · Prices and returns (page F1)

In [ ]:
p = prices("DE")
df = pd.DataFrame({"price": p[1:], "simple": simple_returns(p), "log": log_returns(p)}, index=MONTHS)
print(df.round(4).head(6))
print("\nSum of simple returns :", round(sum(df.simple), 4))
print("Sum of log returns    :", round(sum(df.log), 4), "= ln(last / first) =", round(float(np.log(p[-1] / p[0])), 4))
print("Compounded simple     :", round(compound(df.simple), 4), "= last / first - 1 =", round(p[-1] / p[0] - 1, 4))

In [ ]:
# Gain 50%, then lose 50%: not break-even.
print("after +50% then -50%:", round(100 * 1.5 * 0.5, 1), "per 100 invested")
# Excess return: what you earned over cash.
ex = excess(RETURNS["DE"])
print("DE average monthly return:", round(mean(RETURNS["DE"]) * 100, 2), "%   excess over cash:", round(mean(ex) * 100, 2), "%")

**Try it.** Change `"DE"` to `"CAT"` or `"MKT"` and rerun. Which has the best compounded return? Is it also the one with the best average monthly return?

## 2 · Risk (page F2)

In [ ]:
for t in ["DE", "CAT", "MKT"]:
    pr, r = prices(t), RETURNS[t]
    d, a, b = max_drawdown(pr)
    print(f"{t:4} return {compound(r):6.1%}   volatility {annual_vol(r):5.1%}   worst fall {d:6.1%} ({MONTHS[a-1] if a else 'Sep 24'} to {MONTHS[b-1]})   Sharpe {sharpe(r):5.2f}")

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)
for t in ["DE", "CAT", "MKT"]:
    pr = prices(t)
    ax[0].plot(range(25), [100 * v / pr[0] for v in pr], label=t)
    ax[1].plot(range(25), drawdowns(pr), label=t)
ax[0].set_ylabel("value of 100"); ax[0].legend(); ax[1].set_ylabel("below the peak"); ax[1].set_xlabel("month since Sep 24")
plt.tight_layout(); plt.show()

## 3 · Benchmarks and factors (page F3)

In [ ]:
for t in ["DE", "CAT"]:
    a, b, r2 = alpha_beta(RETURNS[t])
    print(f"{t}: beta {b:.2f}   alpha {a*12:+.1%} a year   R squared {r2:.2f}")

In [ ]:
# Add the three factor benchmarks: a regression on several columns at once (numpy least squares).
X = np.column_stack([np.ones(24), excess(MARKET), SIZE, VALUE, MOMENTUM])
for t in ["DE", "CAT"]:
    coef, *_ = np.linalg.lstsq(X, np.array(excess(RETURNS[t])), rcond=None)
    print(t, dict(zip(["alpha/month", "market", "size", "value", "momentum"], [round(float(c), 3) for c in coef])))

## 4 · Financial and macro data (page F4)

In [ ]:
print(pd.DataFrame({t: valuation(t) for t in ["DE", "CAT"]}).round(3))
print()
s = "Nonfarm payrolls, change (thousands)"
for day in ["2026-09-03", "2026-09-10", "2026-10-05"]:
    print(day, "->", known_on(s, day))

**Try it.** Which date first returns a number? Which returns the *revised* one? A model trained on the revised series but used on the day of the first release has looked into the future.

## 5 · The statistics you need (page F5)

In [ ]:
x, y = MARKET[:-1], RETURNS["DE"][1:]        # this month's market return vs next month's DE return
print("correlation:", round(corr(x, y), 3))
i, s_, r2 = ols(x, y)
print(f"line: next-month DE = {i:.3f} + {s_:.2f} x market   R squared {r2:.3f}")

In [ ]:
# Fit on the early months, judge on the later ones. Never shuffle a time series.
xt, xe = split_by_time(x); yt, ye = split_by_time(y)
i, s_, _ = ols(xt, yt)
mse = lambda xs, ys: mean([(b - i - s_ * a) ** 2 for a, b in zip(xs, ys)])
print("train error:", round(mse(xt, yt), 5), "  test error:", round(mse(xe, ye), 5))

**Try it.** Fit a degree-5 polynomial with `np.polyfit(xt, yt, 5)` and compare train and test error again. That gap is overfitting, and it is the reason chapters 4 and 5 validate through time.